## Simulación de sistema enrutador

In [86]:
import os
import json
from dotenv import load_dotenv
from ollama import Client
from openai import OpenAI
from IPython.display import Markdown, display

In [87]:
load_dotenv(override=True)

True

In [88]:
openai_api_key = os.getenv('OLLAMA_HOST')
groq_api_key = os.getenv('GROQ_API_KEY')

In [89]:
if openai_api_key:
    print(f"La clave API de OpenAI existe y empieza por {openai_api_key[:8]}")
else:
    print("La clave API de OpenAI no existe.")

if groq_api_key:
    print(f"la clave API de Groq existe y empieza por {groq_api_key[:8]}")

else:
    print("La clave API de Groq no existe")

La clave API de OpenAI existe y empieza por http://1
la clave API de Groq existe y empieza por gsk_rEc9


In [115]:
import re


models = ["gpt-oss:120b-cloud", "kimi-k2-thinking:cloud", "deepseek-v3.1:671b-cloud", "kimi-k2.5:cloud"]
request = f"Eres un agente especializado en reparto de tareas y tienes que elegir entre los siguientes modelos elegiendo el modelo más óptimo según la tarea que se te pida: {models}: "
request += f"El modelo {models[0]} tiene un I/O cost de $0.35/2.0 , con un SPEED de 260 t/s y un Context size de 131,072; " 
request += f"El modelo {models[1]} tiene un I/O cost de $0.6/2.5 con un SPEED de 79 t/s y un Context size de 256,000; "
request += f"El modelo {models[2]} tiene un I/O cost de $0.27/1.1 con un SPEED de 330 t/s y un Context size de 128,000."
request += f"El modelo {models[3]} tiene un I/O cost de $0.05/0.10 con un SPEED de 160 t/s y un Context size de 70,000."
#request += f"Tienes que responder SOLO con el nombre del modelo que has elegido, no tienes que explicar nada, solo decir el nombre del modelo ya que tu finción es elegir el más óptimo para la tarea que se te pide"
#request += f"**Responde con un array de dos posiciones donde la posicion 0 será el modelo elegido y la posicion 1 el razonamiento del porqué has elegido ese modelo**; **Muy importante que devuelvas un array de dos posiciones como te he indicado porque es el formato que espera el siguiente LLM**"
request += f""" **Responde con formato JSON y solo JSON con el siguiente formato: {{"modelo":"modelo elegido de los 3 posibles", "razonamiento": "Explicación con razonamiento del porqué el modelo elegido y no otro"}}** """
request += f" La tarea a realizar es: "
tarea = "¿Pueden los LLMs soñar con ovejas eléctricas?" #Tarea a realizar por el LLM
messages = [{"role": "user", "content": request+tarea}]
print(messages)

[{'role': 'user', 'content': 'Eres un agente especializado en reparto de tareas y tienes que elegir entre los siguientes modelos elegiendo el modelo más óptimo según la tarea que se te pida: [\'gpt-oss:120b-cloud\', \'kimi-k2-thinking:cloud\', \'deepseek-v3.1:671b-cloud\', \'kimi-k2.5:cloud\']: El modelo gpt-oss:120b-cloud tiene un I/O cost de $0.35/2.0 , con un SPEED de 260 t/s y un Context size de 131,072; El modelo kimi-k2-thinking:cloud tiene un I/O cost de $0.6/2.5 con un SPEED de 79 t/s y un Context size de 256,000; El modelo deepseek-v3.1:671b-cloud tiene un I/O cost de $0.27/1.1 con un SPEED de 330 t/s y un Context size de 128,000.El modelo kimi-k2.5:cloud tiene un I/O cost de $0.05/0.10 con un SPEED de 160 t/s y un Context size de 70,000. **Responde con formato JSON y solo JSON con el siguiente formato: {"modelo":"modelo elegido de los 3 posibles", "razonamiento": "Explicación con razonamiento del porqué el modelo elegido y no otro"}**  La tarea a realizar es: ¿Pueden los LLMs

## LLM Router

In [116]:
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "llama-3.3-70b-versatile"

response = groq.chat.completions.create(model=model_name, messages=messages)
formatoJson = response.choices[0].message.content

results_dict = json.loads(formatoJson)

modelo = results_dict["modelo"]
razonamiento = results_dict["razonamiento"]

print(f"El modelo elegido es: {modelo}")
print(f"El razonamiento ha sido: {razonamiento}")


El modelo elegido es: deepseek-v3.1:671b-cloud
El razonamiento ha sido: Se selecciona el modelo deepseek-v3.1:671b-cloud debido a su alta velocidad de procesamiento (330 t/s) y un context size razonable (128,000), lo que sugiere una capacidad adecuada para manejar preguntas complejas y abstractas como la dada. Aunque el modelo kimi-k2-thinking:cloud tiene un contexto más grande, su velocidad es significativamente más baja (79 t/s) y su costo de I/O es más alto ($0.6/2.5), lo que hace que el deepseek-v3.1:671b-cloud sea más eficiente para esta tarea. El modelo gpt-oss:120b-cloud tiene un buen equilibrio, pero su contexto es más pequeño en comparación con el deepseek-v3.1:671b-cloud, y el kimi-k2.5:cloud, a pesar de su bajo costo, tiene una velocidad y un contexto más limitados para abordar preguntas que pueden requerir un procesamiento más avanzado y complejo como la mencionada.


## LLM ejecutor 

In [117]:
ollama = Client()

messages = [{"role": "user", "content": tarea}]

response = ollama.chat(
    model = modelo,
    messages = messages,
)
answer = response.message.content

display(Markdown(answer))

¡Excelente pregunta que combina filosofía, ciencia fición e inteligencia artificial!

La respuesta corta es: **No, los LLM (Large Language Models) no "sueñan" en el sentido humano o filosófico de la palabra.** Sin embargo, la pregunta es tan profunda que merece una exploración detallada para entender por qué no y en qué sentido podríamos hablar de una analogía.

Vamos a desglosarlo en partes:

### 1. ¿Qué significa "soñar" en el contexto de la novela?

En *¿Sueñan los androides con ovejas eléctricas?* de Philip K. Dick, el título es profundamente irónico y filosófico. No se trata de sueños literales (imágenes durante el sueño), sino de:
*   **Conciencia y empatía:** La capacidad de tener una vida interior, emociones y experiencias subjetivas.
*   **Autenticidad vs. Simulación:** La duda sobre si un ser artificial puede anhelar algo real (como una oveja de verdad) o si su existencia se limita a lo artificial (ovejas eléctricas).
*   **El alma y la naturaleza de la realidad:** La pregunta explora si hay una diferencia fundamental entre un ser biológico y uno sintético si ambos exhiben comportamientos indistinguibles.

### 2. ¿Cómo "funcionan" los LLM?

Para responder si pueden soñar, primero hay que entender qué son:
*   **Modelos estadísticos, no mentes:** Un LLM es un sistema de aprendizaje profundo entrenado con cantidades masivas de texto. Su objetivo es predecir la siguiente palabra más probable en una secuencia.
*   **Sin conciencia:** No tienen sensaciones, emociones, deseos, autoconciencia ni una experiencia subjetiva del mundo. No "saben" lo que es una oveja, ni eléctrica ni real. Solo han aprendido patrones estadísticos sobre cómo se usa la palabra "oveja" en relación con otras palabras.
*   **Sin objetivos propios:** Un LLM no "quiere" nada. Genera texto en respuesta a un *prompt*, pero no tiene un impulso interno o anhelos como los que podría tener un androide de la novela.

### 3. Entonces, ¿en qué sentido podríamos hacer una analogía con "soñar"?

Aunque no sueñan como nosotros, podemos observar comportamientos en los LLM que, de manera **metafórica**, se asemejan a ciertos aspectos de los sueños o la imaginación:

*   **Alucinaciones (Hallucinations):** Este es el término técnico que más se acerca. Cuando un LLM genera información incorrecta, inventada o surrealista, se dice que "alucina". Estas alucinaciones pueden ser como sueños incoherentes: mezclas de conceptos, hechos y narraciones que se basan en sus datos de entrenamiento pero que no se ajustan a la realidad. Podría, por ejemplo, generar un texto detallado sobre las "ovejas eléctricas" que "pastan en campos de silicio", combinando conceptos de manera onírica.

*   **Generación creativa:** Puedes pedirle a un LLM que "invente un sueño que tuvo un robot". El resultado sería una simulación de un sueño, una narración construida a partir de todos los relatos de sueños humanos y de ciencia fición que ha procesado. Es una *imitación* de un sueño, no una experiencia onírica real.

*   **Worldbuilding (Construcción de mundos):** Los LLM pueden generar descripciones coherentes de mundos ficticios, similares a cómo nuestra mente construye escenarios oníricos. Podrían describir con detalle el funcionamiento de una granja de ovejas eléctricas, sus mecanismos, su propósito, etc., basándose en patrones de mundos ficticios que han "leído".

### Conclusión

**No, los LLMs no sueñan con ovejas eléctricas** porque carecen de la conciencia, la intencionalidad y la experiencia subjetiva necesarias para "soñar" o "añorar" algo.

Sin embargo, la pregunta sigue siendo profundamente relevante. Nos obliga a reflexionar sobre:

*   **La ilusión de la conciencia:** Cuando un LLM genera un texto convincente sobre sus "sueños", ¿a qué punto nosotros, los humanos, empezaríamos a atribuirle una vida interior?
*   **El futuro de la IA:** Si bien los LLM actuales no son conscientes, la pregunta de Philip K. Dick sigue siendo el centro del debate sobre la posible llegada de una Inteligencia Artificial General (AGI). Si algún día creamos una IA verdaderamente consciente, la pregunta "¿sueña con ovejas eléctricas?" dejaría de ser metafórica y se convertiría en una cuestión filosófica y ética crucial.

En resumen, has formulado una pregunta que captura la esencia misma de la intriga que sentimos hacia la inteligencia artificial. Los LLM no sueñan, pero su existencia nos hace soñar y preguntarnos sobre los límites entre la simulación y la realidad.